# Clase 158 — GANs: DCGAN, Progressive GAN, StyleGAN

Un **GAN** (Goodfellow, 2014) enfrenta dos redes: el **Generador** crea muestras
desde ruido y el **Discriminador** distingue real de falso. Implementamos un
**DCGAN** con Keras (`Conv2DTranspose` en G, `Conv2D` en D) y el **loop
adversarial** con doble `GradientTape`. Mencionamos Progressive GAN y StyleGAN.

**Requiere:** `tensorflow` / `keras`. Código = **API real de Keras**; no se
ejecuta sin TF.

## 1. Entorno

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TF = True
    tf.random.set_seed(42)
    print('tensorflow:', tf.__version__)
except Exception as e:
    HAS_TF = False
    print('tensorflow no instalado. Motivo:', type(e).__name__)

import numpy as np
np.random.seed(42)
LATENT = 100

## 2. Generador (DCGAN): `z → 28×28`

`Dense(7·7·128) → reshape → Conv2DTranspose ×2` para upsamplear a 28×28.
`BatchNormalization` estabiliza; salida `tanh` (imágenes en `[-1, 1]`).

In [ ]:
if HAS_TF:
    def make_generator():
        return keras.Sequential([
            keras.Input(shape=(LATENT,)),
            layers.Dense(7 * 7 * 128, use_bias=False),
            layers.BatchNormalization(), layers.LeakyReLU(0.2),
            layers.Reshape((7, 7, 128)),
            layers.Conv2DTranspose(64, 4, strides=2, padding='same', use_bias=False),
            layers.BatchNormalization(), layers.LeakyReLU(0.2),
            layers.Conv2DTranspose(1, 4, strides=2, padding='same',
                                   activation='tanh'),   # -> 28x28x1
        ], name='generator')
    generator = make_generator()
    generator.summary()
else:
    print('G: Dense(7*7*128)->Reshape->Conv2DTranspose(64)->Conv2DTranspose(1,tanh)')

## 3. Discriminador: `28×28 → prob real/fake`

`Conv2D ×2` con `LeakyReLU` + `Dropout`; salida `Dense(1)` (logit).

In [ ]:
if HAS_TF:
    def make_discriminator():
        return keras.Sequential([
            keras.Input(shape=(28, 28, 1)),
            layers.Conv2D(64, 4, strides=2, padding='same'),
            layers.LeakyReLU(0.2), layers.Dropout(0.3),
            layers.Conv2D(128, 4, strides=2, padding='same'),
            layers.LeakyReLU(0.2), layers.Dropout(0.3),
            layers.Flatten(),
            layers.Dense(1),   # logit
        ], name='discriminator')
    discriminator = make_discriminator()
    discriminator.summary()
else:
    print('D: Conv2D(64)->Conv2D(128)->Flatten->Dense(1)')

## 4. Losses y optimizadores

BCE con logits. **Label smoothing**: etiquetas reales = 0.9. Optimizador clásico
DCGAN: `Adam(2e-4, beta_1=0.5)`.

In [ ]:
if HAS_TF:
    bce = keras.losses.BinaryCrossentropy(from_logits=True)

    def d_loss(real_out, fake_out):
        real_l = bce(tf.ones_like(real_out) * 0.9, real_out)   # label smoothing
        fake_l = bce(tf.zeros_like(fake_out), fake_out)
        return real_l + fake_l

    def g_loss(fake_out):
        return bce(tf.ones_like(fake_out), fake_out)   # G quiere que D diga "real"

    g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
    d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
    print('Losses BCE + label smoothing 0.9 ; Adam(2e-4, beta_1=0.5)')
else:
    print('D_loss = BCE(real=0.9) + BCE(fake=0) ; G_loss = BCE(fake->1)')

## 5. `train_step` adversarial (doble `GradientTape`)

Se actualizan D y G en el mismo step con dos cintas independientes.

In [ ]:
if HAS_TF:
    @tf.function
    def train_step(real_images, batch_size):
        noise = tf.random.normal([batch_size, LATENT])
        with tf.GradientTape() as dt, tf.GradientTape() as gt:
            fake = generator(noise, training=True)
            real_out = discriminator(real_images, training=True)
            fake_out = discriminator(fake, training=True)
            dl = d_loss(real_out, fake_out)
            gl = g_loss(fake_out)
        d_grads = dt.gradient(dl, discriminator.trainable_variables)
        g_grads = gt.gradient(gl, generator.trainable_variables)
        d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))
        g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))
        return dl, gl
    print('train_step con doble GradientTape definido.')
else:
    print('Dos GradientTape: una para D (dl) y otra para G (gl), mismo step.')

## 6. Loop de entrenamiento (esqueleto) + diagnóstico

Si `D_loss → 0`, el discriminador ganó y el gradiente de G se desvanece.

In [ ]:
EPOCHS, BATCH = 50, 128
if HAS_TF:
    # (X_train, _), _ = keras.datasets.fashion_mnist.load_data()
    # X_train = (X_train.astype('float32') - 127.5) / 127.5   # a [-1,1]
    # for epoch in range(EPOCHS):
    #     for batch in dataset:
    #         dl, gl = train_step(batch, BATCH)
    print(f'Loop: {EPOCHS} epocas; monitorear D_loss y G_loss por step.')
else:
    print('Alternar updates de D y G; vigilar mode collapse (64 samples iguales).')

## 7. Progressive GAN y StyleGAN (nota)

- **Progressive GAN** (Karras 2018): empieza en 4×4 y sube resolución agregando capas.
- **StyleGAN** (Karras 2019+): mapea `z → w` y lo inyecta vía **AdaIN** en cada
  capa → control de estilo, caras hiperrealistas. Sigue competitivo en 2026.
- Métrica estándar: **FID** (Fréchet Inception Distance); menor = mejor.

## Ejercicios

1. **DCGAN básico**: entrená G y D en Fashion-MNIST y generá un grid 8×8.
2. **Diagnóstico**: graficá `D_loss` y `G_loss` por step; si `D_loss → 0`, G pierde.
3. **Mode collapse**: tras N épocas generá 64 samples; si son iguales → collapse.
4. **WGAN-GP**: reemplazá la loss por Wasserstein-1 + gradient penalty y compará estabilidad.
5. **FID**: reportá el FID del modelo (`tensorflow_gan.eval.fid` o implementación propia).

## Conclusiones

- GAN = juego min-max entre Generador y Discriminador; el equilibrio genera
  muestras indistinguibles de las reales.
- DCGAN usa `Conv2DTranspose` (upsampling en G) y `Conv2D` (downsampling en D).
- El loop adversarial requiere doble `GradientTape`; `Adam(2e-4, beta_1=0.5)` es el clásico.
- Fallos típicos: mode collapse y `D_loss→0`; se mitigan con label smoothing, dropout, WGAN-GP.
- Progressive GAN y StyleGAN escalan a alta resolución; FID mide la calidad.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README de esta clase. El código que usa librerías pesadas (`transformers` / `torch` / `keras` / `diffusers`) es la **API real** de la industria y se valida por sintaxis (los modelos requieren GPU/descarga). Los **núcleos numéricos** están en numpy puro, son **ejecutables** y se autoverifican con `assert`.

### Ejercicio 1 — DCGAN básico (Keras) + GAN de juguete EJECUTABLE en numpy

In [ ]:
# --- API real: loop de entrenamiento DCGAN en Fashion-MNIST ---
if HAS_TF:
    from tensorflow import keras
    import tensorflow as tf
    (Xtr, _), _ = keras.datasets.fashion_mnist.load_data()
    Xtr = (Xtr.astype('float32') - 127.5) / 127.5              # a [-1, 1]
    Xtr = Xtr[..., None]
    ds = tf.data.Dataset.from_tensor_slices(Xtr).shuffle(1000).batch(BATCH)
    for epoch in range(EPOCHS):
        for batch in ds:
            dl, gl = train_step(batch, tf.shape(batch)[0])
    print('DCGAN entrenado; generar grid 8x8 desde z ~ N(0, I).')
else:
    print('Sin TF: abajo, un GAN de juguete 1D 100% ejecutable en numpy puro.')

# --- GAN de JUGUETE ejecutable: gen/disc LINEAL sobre datos 1D ---
import numpy as np
rng = np.random.default_rng(0)
def _sig(s): return 1.0 / (1.0 + np.exp(-s))
REAL_MU, REAL_SD = 4.0, 0.5
wg, bg = 1.0, 0.0            # Generador lineal G(z) = wg*z + bg  (z ~ N(0,1))
a, b = 0.0, 0.0             # Discriminador logistico D(x) = sigmoid(a*x + b)
ema_bg, ema_wg, decay = bg, wg, 0.99
lr, N = 0.05, 256
for step in range(4000):
    # --- paso D: sube log D(real) + log(1 - D(fake)) ---
    xr = rng.normal(REAL_MU, REAL_SD, N)
    z = rng.normal(0, 1, N); xf = wg * z + bg
    Dr, Df = _sig(a * xr + b), _sig(a * xf + b)
    a += lr * (np.mean((1 - Dr) * xr) - np.mean(Df * xf))
    b += lr * (np.mean(1 - Dr) - np.mean(Df))
    # --- paso G: sube log D(fake) (no saturante) ---
    z = rng.normal(0, 1, N); xf = wg * z + bg
    Df = _sig(a * xf + b)
    bg += lr * np.mean((1 - Df) * a)
    wg += lr * np.mean((1 - Df) * a * z)
    ema_bg = decay * ema_bg + (1 - decay) * bg     # EMA del generador (a la StyleGAN)
    ema_wg = decay * ema_wg + (1 - decay) * wg
fake = ema_wg * rng.normal(0, 1, 5000) + ema_bg
print(f'media generada = {fake.mean():.3f}  |  media real = {REAL_MU}')
assert abs(fake.mean() - REAL_MU) < 0.4        # G aprendio la distribucion real
mixed = np.concatenate([rng.normal(REAL_MU, REAL_SD, 2000), fake[:2000]])
assert abs(_sig(a * mixed + b).mean() - 0.5) < 0.15   # D en el equilibrio (~0.5)
print('OK: el GAN de juguete convergio (G ~ real, D ~ 0.5).')

### Ejercicio 2 — Diagnóstico: curvas de `D_loss` y `G_loss`

In [ ]:
if HAS_TF:
    import tensorflow as tf
    d_hist, g_hist = [], []
    # for batch in ds: dl, gl = train_step(batch, BATCH); d_hist.append(float(dl)); g_hist.append(float(gl))
    # plt.plot(d_hist, label='D'); plt.plot(g_hist, label='G'); plt.legend()
    print('Registrar dl, gl por step y graficar. Si D_loss -> 0, D ganó y el'
          ' gradiente de G se desvanece (G deja de aprender).')
else:
    print('D_loss -> 0 : el discriminador domina, G no recibe señal util.')

### Ejercicio 3 — Mode collapse: 64 samples iguales

In [ ]:
import numpy as np
# Deteccion generica de mode collapse: baja varianza entre muestras generadas.
def collapsed(samples, thr=1e-3):
    return float(samples.var(axis=0).mean()) < thr
diverse = np.random.normal(size=(64, 784))
collapse = np.repeat(np.random.normal(size=(1, 784)), 64, axis=0)   # todas iguales
assert not collapsed(diverse) and collapsed(collapse)
print('OK: detector de mode collapse por varianza entre 64 samples.')

### Ejercicio 4 — WGAN-GP: gradient penalty

In [ ]:
if HAS_TF:
    import tensorflow as tf
    def gradient_penalty(critic, real, fake):
        eps = tf.random.uniform([tf.shape(real)[0], 1, 1, 1])
        inter = eps * real + (1 - eps) * fake            # punto interpolado
        with tf.GradientTape() as t:
            t.watch(inter)
            pred = critic(inter, training=True)
        grads = t.gradient(pred, inter)
        norm = tf.sqrt(tf.reduce_sum(grads ** 2, axis=[1, 2, 3]) + 1e-12)
        return tf.reduce_mean((norm - 1.0) ** 2)         # penaliza ||grad|| != 1
    print('WGAN-GP: critic sin sigmoid; loss = E[fake]-E[real] + λ·GP (λ=10).')
else:
    print('WGAN-GP: distancia de Wasserstein + penalizacion de gradiente'
          ' (||grad||=1). Mucho mas estable que DCGAN, casi sin mode collapse.')

### Ejercicio 5 — FID (Fréchet Inception Distance)

In [ ]:
import numpy as np
# FID entre dos gaussianas de features: ||mu_r-mu_g||^2 + Tr(Cr+Cg-2*sqrt(Cr*Cg)).
from scipy.linalg import sqrtm
def fid(feat_real, feat_fake):
    mr, mg = feat_real.mean(0), feat_fake.mean(0)
    Cr, Cg = np.cov(feat_real, rowvar=False), np.cov(feat_fake, rowvar=False)
    covmean = sqrtm(Cr @ Cg)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(((mr - mg) ** 2).sum() + np.trace(Cr + Cg - 2 * covmean))
rng = np.random.default_rng(0)
a1 = rng.normal(0, 1, (500, 16))
assert fid(a1, a1) < 1e-6                    # misma distribucion -> FID ~ 0
assert fid(a1, rng.normal(3, 1, (500, 16))) > fid(a1, rng.normal(0.2, 1, (500, 16)))
print('OK: FID=0 para distribuciones identicas y crece con la distancia.')
print('En real: features de InceptionV3 (pool3, 2048-d) sobre reales vs generadas.')